# L4: Verification Loop

In Lesson 3, the agent decided for itself when it was finished. But an agent that says "done" isn't necessarily right. In this lesson you'll add a second role, a **grader**, that checks the agent's work against a rubric and sends it back for revision until the work meets the bar.

**By the end of this lesson you will be able to:**
- Explain why an agent's own claim of "done" isn't enough
- Read a rubric and see how it turns "good work" into checkable criteria
- Describe the verification loop: draft, grade, feedback, revise, grade again
- Interpret the grader's verdicts and per-criterion results
- Explain what stops the loop, and why those limits matter

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, Markdown, Image

from deepagent_helpers import (DEFAULT_RUBRIC, GRADER_PROMPT,
    langsmith_status, get_ticket,
    build_review_agent, run_verification_loop_live,
    save_verification_trace, load_verification_trace,
)
from visualization_helpers import (
    render_review_trace, render_verification_summary,
    render_grading_grid, render_revision_progression,render_grader_feedback,
    render_report, render_report_progression, render_agent_middleware,
)

print("✓ Environment ready")

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> file:</b> 1) click on the <em>"View"</em> option on the top menu of the notebook and then 2) click on <em>"File Browser"</em>.</p>

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download"</em>.</p>
</div>

## The Rubric


A rubric is the list of standards the grader uses to judge the agent's work. Good criteria are specific enough that a reviewer can check them one by one and point to evidence.

Print the rubric below. It has three criteria, and each is a concrete check, not a vague quality:
- **completeness:** every clause of the requirement is addressed
- **clarity:** each ambiguity states what fact or decision would resolve it (a `Resolved by:` line)
- **depth:** each ambiguity names a concrete production failure it could cause (an `Edge case:` line)

Notice that clarity and depth are not about style. They are about whether specific lines exist and say something useful.

In [ ]:
display(Markdown("**The rubric:**\n\n" + DEFAULT_RUBRIC))

## Building the Verification Loop

The agent now runs inside a loop with a grader:

1. The agent writes its analysis, as in Lesson 3
2. The grader reads the file and checks it against the rubric
3. If every criterion passes, the loop ends
4. If any criterion fails, the grader's feedback goes back to the agent, which revises the file, and the grader checks again

Three settings control this:
- `MODEL`: the model that does the work
- `GRADER_MODEL`: the model that judges it. It can differ from the worker, and keeping the writer and the judge separate is a common design. Here both use the same model.
- `MAX_RUBRIC_ITERATIONS`: caps how many times the work can be graded, so the loop can't revise forever

In [ ]:
MODEL = "openai:gpt-5.4-mini"
GRADER_MODEL = "openai:gpt-5.4-mini"
MAX_RUBRIC_ITERATIONS = 3

agent = build_review_agent(
    model=MODEL,
    grader_model=GRADER_MODEL,
    rubric_iterations=MAX_RUBRIC_ITERATIONS,   
    max_turns=40,
)
print("✓ Verification loop agent created")

## Run the Agent

This lesson uses a new ticket with six requirement clauses. Almost every clause contains a vague word, such as "timely", "appropriate", "efficiently", "easily", "immediate", or "reliably". That gives the grader plenty to check.

In [ ]:
TICKET_ID = "NOTIF-205"
requirement_doc = get_ticket(TICKET_ID)

print(f"Ticket {TICKET_ID}:\n")
print(requirement_doc)

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different Run Results:</b> LLMs do not always produce the same results. Consequently, the agent behavior you experience when running labs may differ from the video.</p>

##### Run the verification loop

The next cell has a switch, `USE_LIVE_LLM`:
- `True`: runs the agent and grader live. This takes longer than L3, because every grading pass is another model call.
- `False`: loads a saved trace from a previous real run.

> If you see a message that LangSmith tracing is off, it only means your own run isn't being recorded in LangSmith. Everything else works as normal.

The cell shows two results:

**1. The summary**
- **Grader verdicts:** the verdict from each grading pass, in order. `needs_revision → satisfied` means the first draft was rejected and the revision passed.
- **Turns:** how much work the run took: model turns, steps, and grading passes
- **Stopped because:** why the loop ended

**2. The grading grid**
- One row per rubric criterion and one column per grading pass
- The bottom rows show the overall verdict and the file size after each pass

Read the grid left to right: which criteria failed in Pass 0, and did they pass in Pass 1? The file size row also tells you something. The file grows after revision, so the agent is adding the missing content.

Exact turn counts and file sizes vary from run to run.

In [ ]:
USE_LIVE_LLM = True
if USE_LIVE_LLM:
    run = run_verification_loop_live(agent, ticket_id=TICKET_ID, 
                                     rubric=DEFAULT_RUBRIC)
    save_verification_trace(run, "lesson_4_live_trace.json")
    print(langsmith_status())
else:
    run = load_verification_trace("lesson_4_prebaked_trace.json")

display(render_verification_summary(run))
display(render_grading_grid(run))

### Execution Trace

Click on the following link to view saved agent execution trace.

🔗   [[ Agent's Trace ]](https://smith.langchain.com/public/9391a9d7-0f2b-4437-adce-924dbeae397e/t?start_time=2026-10-06T08%3A50%3A45.681563Z&run_id=01a11068-f853-7441-a97c-293256873465&conversationTab=trace&trace_id=01a11068-7d31-70f1-b28a-0de12f7bff56&scroll_to=output)

As you look through it, find the point where the agent finishes its first draft and the grader takes over, then follow what happens next.

In [ ]:
display(render_grader_feedback(run))

##### What the grader said

For each pass, the grader gives you:
- A short **overall verdict** in plain language
- A ✅ or ❌ for **every rubric criterion**
- For each ❌, a **gap**: what is missing and what to add

## The Final Report

The next cell shows three panels:
1. The **original requirement**
2. The agent's **first draft** (Pass 0), which the grader rejected
3. The **final version**, which the grader accepted

Compare the **Ambiguities** section in panels 2 and 3. The rubric checks that section for the extra lines, so that is where you should see the change. The other sections should look much the same.

In [ ]:
display(render_report_progression(requirement_doc, 
                              run.grading[0].analysis_markdown,
                              run.analysis_markdown))

## Conclusion

In this lesson you added a verification loop on top of the agent loop from Lesson 3.

**Key takeaways**
- **"Done" isn't the same as "good."** An agent can finish a task and still miss what you needed. A separate check catches that.
- **A rubric makes quality checkable.** Each criterion was a concrete test, so the grader could say pass or fail for each one.
- **Specific feedback drives revision.** The grader named the gap and what to add, so the agent knew exactly what to fix.
- **Revisions can be targeted.** Only the failing part of the report changed. The agent improved the Ambiguities section without rewriting the rest.
- **Separate the worker from the judge.** The grader runs as its own role, which keeps the check independent of the work.
- **Loops need limits.** The loop stops when the rubric is satisfied or when the iteration limit is reached, so it can't run forever.
- **The rubric sets the ceiling.** The report can only be as good as the criteria it is graded against. Verification also costs extra model calls, so it is a trade-off between quality and cost.

**Try it yourself**
- Edit the rubric: make a criterion stricter or add a new one, then rerun and compare the grading grid
- Lower `MAX_RUBRIC_ITERATIONS` and see how the outcome changes when the agent has fewer chances to revise